# 03 — TF-IDF + LinearSVC (M2)

**Objetivo:** Treinar o classificador M2 (TF-IDF + LinearSVC com CalibratedClassifierCV) no split de treino balanceado e avaliar no split de validacao.

**Dependencias:** Notebook 01 executado (splits em `artifacts/splits/`).

In [ ]:
import pandas as pd

from economy_classifier.tfidf import train_tfidf_classifier, TfidfTrainingConfig
from economy_classifier.evaluation import compute_binary_metrics, compute_roc_auc
from economy_classifier.project import (
    SPLITS_DIR, create_run_directory, build_run_metadata, persist_run_artifacts,
)
from economy_classifier.visualization import (
    configure_style, plot_confusion_matrix, plot_roc_curve, plot_pr_curve, save_figure,
)

In [ ]:
configure_style()

balanced_train = pd.read_parquet(SPLITS_DIR / 'balanced_train.parquet')
val = pd.read_parquet(SPLITS_DIR / 'val.parquet')

print(f'Treino balanceado: {len(balanced_train):,} linhas')
print(f'Validacao:         {len(val):,} linhas')

## Treino

O LinearSVC nao produz probabilidades nativas. `CalibratedClassifierCV(cv=3)` treina o classificador 3 vezes internamente para calibrar scores via Platt scaling.

In [ ]:
config = TfidfTrainingConfig(classifier='linearsvc')
run_dir = create_run_directory('tfidf-training', 'linearsvc')

result = train_tfidf_classifier(balanced_train, val, run_dir=run_dir, config=config)

print(f'Vocabulario: {result["vocabulary_size"]:,} termos')
print(f'Treino: {result["timing"]["train_seconds"]:.1f}s')
print(f'Inferencia: {result["timing"]["inference_seconds"]:.1f}s')

## Avaliacao no split de validacao

In [ ]:
preds = result['predictions']
metrics = compute_binary_metrics(preds['y_true'].values, preds['y_pred'].values)
metrics['auc_roc'] = round(compute_roc_auc(preds['y_true'].values, preds['y_score'].values), 4)

for k, v in metrics.items():
    print(f'  {k:<12}: {v:.4f}')

## Figuras

In [ ]:
fig_cm = plot_confusion_matrix(preds['y_true'], preds['y_pred'], title='M2 LinearSVC — Matriz de Confusao (val)')
save_figure(fig_cm, run_dir / 'figures', 'linearsvc_confusion_matrix')

fig_roc = plot_roc_curve(preds['y_true'], preds['y_score'], title='M2 LinearSVC — Curva ROC (val)')
save_figure(fig_roc, run_dir / 'figures', 'linearsvc_roc_curve')

fig_pr = plot_pr_curve(preds['y_true'], preds['y_score'], title='M2 LinearSVC — Curva PR (val)')
save_figure(fig_pr, run_dir / 'figures', 'linearsvc_pr_curve')

## Persistencia

In [ ]:
preds.to_csv(run_dir / 'predictions.csv', index=False)

metadata = build_run_metadata(
    run_dir=run_dir, stage='tfidf-training',
    parameters=config.to_dict(),
    inputs={'balanced_train_rows': len(balanced_train), 'val_rows': len(val)},
    outputs={'model_dir': result['model_dir'], 'vocab_size': result['vocabulary_size']},
    summary={'method': 'linearsvc', **metrics},
    timing=result['timing'],
)
persist_run_artifacts(run_dir=run_dir, metadata=metadata, metrics=metrics)
print(f'Artefatos salvos em: {run_dir}')